# Fraud Detection — Credit Card Transactions
Binary classification model to predict whether a credit card transaction is fraudulent or not using a Random Forest Classifier.

In [17]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.impute import SimpleImputer

In [26]:
df = pd.read_csv(r'C:\Users\PL\Documents\ML data\fraudTest.csv')

In [23]:
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0


### Data Preprocessing and Feature Engineering
The original dataset contained 142,681 records, with only 0.39% of transactions labeled as fraudulent. To address this severe class imbalance, undersampling was applied before splitting the data — retaining all fraudulent transactions and sampling 2,500 non-fraudulent records to create a more balanced dataset for training and evaluation.
Additionally, an age feature was engineered by calculating the difference between trans_date_trans_time and dob, representing the cardholder's age at the time of each transaction. This feature was included alongside the other predictors to improve the model's ability to detect fraud.

In [28]:
fraud = df[df['is_fraud']==1]
not_fraud = df[df['is_fraud']==0].sample(n=2500, random_state=42)
df_1 = pd.concat([fraud,not_fraud])

In [29]:
df_1['trans_date_trans_time'] = pd.to_datetime(df_1['trans_date_trans_time'])
df_1['dob'] = pd.to_datetime(df['dob'])
df_1['age'] = (df_1['trans_date_trans_time'] - df_1['dob']).dt.days / 365

In [30]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4645 entries, 1685 to 142681
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Unnamed: 0             4645 non-null   int64         
 1   trans_date_trans_time  4645 non-null   datetime64[ns]
 2   cc_num                 4645 non-null   int64         
 3   merchant               4645 non-null   object        
 4   category               4645 non-null   object        
 5   amt                    4645 non-null   float64       
 6   first                  4645 non-null   object        
 7   last                   4645 non-null   object        
 8   gender                 4645 non-null   object        
 9   street                 4645 non-null   object        
 10  city                   4645 non-null   object        
 11  state                  4645 non-null   object        
 12  zip                    4645 non-null   int64         
 13  lat

In [31]:
df_1.describe()

,Unnamed: 0,trans_date_trans_time,cc_num,amt,zip,lat,long,city_pop,dob,unix_time,merch_lat,merch_long,is_fraud,age
count,4645.000000,4645,4.645000e+03,4645.000000,4645.000000,4645.000000,4645.000000,4.645000e+03,4645,4.645000e+03,4645.000000,4645.000000,4645.000000,4645.000000
mean,261000.882239,2020-09-26 04:22:55.467169024,4.075796e+17,277.792351,48290.513886,38.777540,-90.268006,7.290833e+04,1972-10-26 17:09:51.345532832,1.380169e+09,38.775908,-90.271928,0.461787,47.947631
min,53.000000,2020-06-21 12:30:39,6.041621e+10,1.010000,1257.000000,20.027100,-165.672300,2.300000e+01,1924-10-30 00:00:00,1.371818e+09,19.161782,-166.509533,0.000000,15.468493
25%,134726.000000,2020-08-07 15:03:32,3.854431e+13,19.620000,24970.000000,34.957200,-96.743000,8.250000e+02,1961-04-25 00:00:00,1.375888e+09,34.908556,-96.809288,0.000000,34.183562
50%,260772.000000,2020-09-25 12:50:22,3.506043e+15,78.000000,48088.000000,39.609400,-87.764400,2.501000e+03,1974-07-19 00:00:00,1.380113e+09,39.609781,-87.708217,0.000000,46.249315
75%,385961.000000,2020-11-18 23:34:01,4.518351e+15,346.420000,70466.000000,42.078900,-80.128400,1.474200e+04,1986-06-11 00:00:00,1.384818e+09,42.080129,-80.085263,1.000000,59.353425
max,555634.000000,2020-12-31 23:28:13,4.992346e+18,2507.650000,99921.000000,64.755600,-67.950300,2.906700e+06,2005-01-29 00:00:00,1.388532e+09,64.475811,-66.960745,1.000000,95.827397
std,149146.197199,NaN,1.303280e+18,361.019285,26751.876487,5.075715,13.841134,2.441056e+05,NaN,4.907444e+06,5.102498,13.841231,0.498591,17.519876


In [32]:
df_1.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,age
1685,1685,2020-06-21 22:06:39,3560725013359375,fraud_Hamill-D'Amore,health_fitness,24.84,Brooke,Smith,F,63542 Luna Brook Apt. 012,...,-102.7413,23,Cytogeneticist,1969-09-15,16bf2e46c54369a8eab2214649506425,1371852399,32.575873,-102.604290,1,50.800000
1767,1767,2020-06-21 22:32:22,6564459919350820,"fraud_Rodriguez, Yost and Jenkins",misc_net,780.52,Douglas,Willis,M,619 Jeremy Garden Apt. 681,...,-90.3508,1306,Public relations officer,1958-09-10,ab4b379d2c0c9c667d46508d4e126d72,1371853942,42.461127,-91.147148,1,61.821918
1781,1781,2020-06-21 22:37:27,6564459919350820,fraud_Nienow PLC,entertainment,620.33,Douglas,Willis,M,619 Jeremy Garden Apt. 681,...,-90.3508,1306,Public relations officer,1958-09-10,47a9987ae81d99f7832a54b29a77bf4b,1371854247,42.771834,-90.158365,1,61.821918
1784,1784,2020-06-21 22:38:55,4005676619255478,"fraud_Heathcote, Yost and Kertzmann",shopping_net,1077.69,William,Perry,M,458 Phillips Island Apt. 768,...,-90.9027,71335,Herbalist,1994-05-31,fe956c7e4a253c437c18918bf96f7b62,1371854335,31.204974,-90.261595,1,26.076712
1857,1857,2020-06-21 23:02:16,3560725013359375,fraud_Hermann and Sons,shopping_pos,842.65,Brooke,Smith,F,63542 Luna Brook Apt. 012,...,-102.7413,23,Cytogeneticist,1969-09-15,f6838c01f5d2262006e6b71d33ba7c6d,1371855736,31.315782,-102.736390,1,50.800000


In [33]:
X = df_1[[
    'category',
    'amt',
    'gender',
    'city_pop',
    'age'
]]

y = df_1['is_fraud']

In [34]:
X.head()

,category,amt,gender,city_pop,age
1685,health_fitness,24.84,F,23,50.800000
1767,misc_net,780.52,M,1306,61.821918
1781,entertainment,620.33,M,1306,61.821918
1784,shopping_net,1077.69,M,71335,26.076712
1857,shopping_pos,842.65,F,23,50.800000


### Feature Categorization and Pipeline
Numerical and categorical features were separated into their respective groups to prepare them for transformation.
A ColumnTransformer pipeline was built with the following:

Numeric features (amt, city_pop, age): Missing values imputed using median — less sensitive to extreme values. Scaled with RobustScaler due to severe outliers identified in df_1.describe().
Categorical features (category, gender): Missing values imputed using most_frequent. Encoded with OneHotEncoder(handle_unknown='ignore') to handle unseen categories during prediction without errors.

In [35]:
numeric_features = ['amt','city_pop','age']
categorical_features = ['category','gender']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced'))
])

In [36]:
X_train, X_test, y_train, y_test = train_test_split (
    X, y, test_size = 0.25, random_state = 42, stratify = y )

### Hyperparameter Tuning with GridSearchCV

For hyperparameter tuning, we used `GridSearchCV` to determine the optimal number of trees for the model. The `param_grid` tested `n_estimators` values of `100`, `200`, `300`, `400`, and `500` within a pipeline that utilized a `RandomForestClassifier` with `class_weight='balanced'` to better handle the class imbalance problem.

We applied `cv=5` for cross-validation and selected `recall` as the scoring metric because, in fraud detection, failing to identify fraudulent transactions (false negatives) is more costly than incorrectly flagging legitimate transactions (false positives).

After evaluation, `GridSearchCV` identified `500` estimators as the best-performing parameter for the model.

In [37]:
param_grid = {'model__n_estimators': [100,200,300,400,500]}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='recall'
)

grid_search.fit(X_train,y_train)

best_param = grid_search.best_params_
print(best_param)

{'model__n_estimators': 400}


In [38]:
best_estimator = grid_search.best_estimator_

In [62]:
probs = best_estimator.predict_proba(X_test)[:,1]
preds = (probs >= 0.20).astype(int)

# Threshold Tuning
Thresholds from 0.50 down to 0.10 were tested to find the optimal balance between false negatives and false positives. The final threshold was set at 0.20.
At the baseline threshold of 0.50, the model produced 26 false negatives and 22 false positives. At 0.20, false negatives dropped to 6 at the cost of increasing false positives to 58. This tradeoff is acceptable — undetected fraud directly harms customers and carries a higher long-term business cost than routing additional transactions to manual review.

In [63]:
print(confusion_matrix(y_test, preds))

[[567  58]
 [  6 531]]


In [41]:
feature_importances_df = pd.DataFrame({'features': best_estimator.named_steps['preprocessor'].get_feature_names_out(),
              'importances': best_estimator.named_steps['model'].feature_importances_})

### Feature Importance Analysis

Since `RandomForestClassifier` does not provide coefficients like `LogisticRegression`, we used `feature_importances_` to evaluate how much each feature contributed to the model’s predictions. Feature importance serves as the equivalent interpretation metric for Random Forest models, indicating the relative influence of each feature in determining whether a transaction was classified as fraudulent or not.

Using the feature importance output, we identified the main drivers behind the model’s predictions. The most influential feature was `amount`, contributing approximately **66.8%** to the model’s decision-making process. This was followed by the engineered feature `age` at **7.06%**, and `city_pop` at **7.01%**. These three features were identified as the top contributors influencing fraud detection predictions.

In [42]:
print(feature_importances_df.sort_values(by='importances', ascending=False))

                        features  importances
0                       num__amt     0.664869
2                       num__age     0.071839
1                  num__city_pop     0.069971
14    cat__category_shopping_net     0.031359
5    cat__category_gas_transport     0.026003
7      cat__category_grocery_pos     0.019691
4      cat__category_food_dining     0.014148
11        cat__category_misc_net     0.012810
9             cat__category_home     0.012216
15    cat__category_shopping_pos     0.012203
16          cat__category_travel     0.009983
6      cat__category_grocery_net     0.009360
12        cat__category_misc_pos     0.007571
10       cat__category_kids_pets     0.007538
3    cat__category_entertainment     0.007455
18                 cat__gender_M     0.006883
17                 cat__gender_F     0.006239
13   cat__category_personal_care     0.005745
8   cat__category_health_fitness     0.004118
